# Phase 3 — Threat Hunting (Kaggle Demo)
Notebook này dùng output CTI của Phase 2 để tạo hunting plan, chạy trên sample Sysmon/Windows logs và tính Precision/Recall/F1.


In [ ]:
from pathlib import Path
import sys, json

# Nếu đã upload/unzip repo vào /kaggle/working, cell này sẽ tự tìm root.
candidates = list(Path('/kaggle/working').rglob('Recommandation-CTI-System-main')) if Path('/kaggle/working').exists() else []
ROOT = candidates[0] if candidates else Path.cwd()
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)


In [ ]:
from threat_hunting.pipeline import run_hunting_pipeline

cti_path = ROOT / 'api_backend/mock_data/report_1.json'
log_path = ROOT / 'hunting_lab/sample_logs/windows_sysmon_demo.jsonl'
output_path = ROOT / 'hunting_lab/output/kaggle_phase3_result.json'

with open(cti_path, encoding='utf-8') as f:
    cti = json.load(f)

result = run_hunting_pipeline(cti, str(log_path), str(output_path), evaluate=True)
print('Technique:', result['hunting_plan']['technique_id'])
print('Hypothesis:', result['hunting_plan']['hypothesis'])
print('Telemetry:', *result['hunting_plan']['telemetry'], sep='\n- ')
print('Matched events:', result['matched_events'], '/', result['events_scanned'])
print('Metrics:', {k: result['evaluation'][k] for k in ['precision','recall','f1','false_positive_rate']})


In [ ]:
print(result['hunting_plan']['sigma_rule'])
print('KQL reference:\n', result['hunting_plan']['queries']['kql_reference'])
print('SPL reference:\n', result['hunting_plan']['queries']['spl_reference'])


In [ ]:
# Xem suspicious events
for finding in result['findings']:
    print('---')
    print('Risk:', finding['risk_score'])
    print('Reasons:', finding['reasons'])
    print('Event:', json.dumps(finding['event'], ensure_ascii=False, indent=2))
